<a href="https://colab.research.google.com/github/MaxKnap02/BiasedNewsGenerator/blob/modelFineTuningColab/nb/FinetuningLaMiniGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q bitsandbytes datasets transformers accelerate peft
import torch
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Läuft auf:", device)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# %%
model_name = "MBZUAI/LaMini-GPT-1.5B"  # Reasoning-fähiges 1.5B-Modell
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Lade Basismodell in 4-Bit ...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Falls noch kein pad_token_id vorhanden, auf eos_token_id setzen
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

Lade Basismodell in 4-Bit ...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/984 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/6.28G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.28G [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

In [ ]:
# ==========================
# 3) Vorbereitung für LoRA-Finetuning
# ==========================
model = prepare_model_for_kbit_training(base_model)
model.gradient_checkpointing_enable()  # Memory sparen

# LoRA-Konfiguration:
# - r=16 (Größe der Low-Rank-Matrizen)
# - lora_dropout=0.1 (etwas höher, da Datensatz klein -> Overfitting reduzieren)
# - target_modules: an die GPT-Architektur anpassen (hier: q_proj, k_proj, etc.)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    target_modules=["c_attn", "c_proj", "c_fc"],
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 19,660,800 || all params: 1,577,273,600 || trainable%: 1.2465


In [ ]:
for name, module in model.named_modules():
    print(name)


base_model
base_model.model
base_model.model.transformer
base_model.model.transformer.wte
base_model.model.transformer.wpe
base_model.model.transformer.drop
base_model.model.transformer.h
base_model.model.transformer.h.0
base_model.model.transformer.h.0.ln_1
base_model.model.transformer.h.0.attn
base_model.model.transformer.h.0.attn.c_attn
base_model.model.transformer.h.0.attn.c_attn.base_layer
base_model.model.transformer.h.0.attn.c_attn.lora_dropout
base_model.model.transformer.h.0.attn.c_attn.lora_dropout.default
base_model.model.transformer.h.0.attn.c_attn.lora_A
base_model.model.transformer.h.0.attn.c_attn.lora_A.default
base_model.model.transformer.h.0.attn.c_attn.lora_B
base_model.model.transformer.h.0.attn.c_attn.lora_B.default
base_model.model.transformer.h.0.attn.c_attn.lora_embedding_A
base_model.model.transformer.h.0.attn.c_attn.lora_embedding_B
base_model.model.transformer.h.0.attn.c_attn.lora_magnitude_vector
base_model.model.transformer.h.0.attn.c_proj
base_model.model.

In [ ]:
# z.B. "Yonekin/nius_reasoning"
dataset_name = "Yonekin/nius_reasoning"
ds = load_dataset("json", data_files="https://huggingface.co/datasets/Yonekin/nius_reasoning/resolve/main/qa_sheet_cleaned2.json")

# Künstliche Aufteilung in Train/Val: 80% zu 20%
ds = ds["train"].train_test_split(test_size=0.2, seed=42)
train_ds = ds["train"]
val_ds = ds["test"]

print("Train set size:", len(train_ds))
print("Val set size:", len(val_ds))

Train set size: 612
Val set size: 154


In [ ]:
'''
# ==========================
# 5) Tokenize-Funktion mit Truncation & Padding
# ==========================
# Wir wollen Reasoning + Artikel gleich wichtig nehmen.
# Format => "REASONING:\n<reasoning>\n\nARTICLE:\n<output>"

MAX_LENGTH = 1024  # z.B. 1024 Tokens

def tokenize_fn(example):
    prompt = f"PROMPT:\n{example['input']}\n\nREASONING:\n{example['reasoning']}\n\nARTICLE:\n{example['output']}"

    # 1) Tokenisierung mit PADDING + TRUNCATION auf max_length
    enc = tokenizer(
        prompt,
        max_length=MAX_LENGTH,
        padding="max_length",   # <<< WICHTIG: Feste Länge pro Beispiel
        truncation=True,
        return_tensors="pt"
    )

    # input_ids + attention_mask haben nun bereits die Länge MAX_LENGTH
    input_ids = enc["input_ids"][0].tolist()
    attention_mask = enc["attention_mask"][0].tolist()

    # 2) Prompt-Teil maskieren => -100 in Labels
    #    Falls du prompt nicht in den Loss nehmen willst.
    #    Wir zählen, wie viele "echte" Tokens der Prompt hat (via attention_mask).
    #    ABER: das Problem: padding="max_length" macht den Prompt ggf. auch 1024 Tokens lang.
    #    -> Wir müssen die tatsächlichen Token zählen, NICHT nur len(...)!
    prompt_enc = tokenizer(
        f"PROMPT:\n{example['input']}",
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    # Hier Anzahl 'echter' Tokens in 'prompt_enc' ermitteln
    # (die "1"er in der attention_mask).
    prompt_len = prompt_enc["attention_mask"][0].sum().item()
    # .sum() zählt die Anzahl Tokens != PAD innerhalb des Promptteils.

    # 3) Erstelle Labels => Kopie von input_ids
    labels = input_ids[:]  # Listen-Kopie
    # Alle Prompt-Tokens werden auf -100 gesetzt,
    # so dass nur Reasoning+Artikel in den Loss eingehen
    for i in range(int(prompt_len)):
        labels[i] = -100

    # (Optional) Nochmal: wir haben JA ALLES schon 'max_length' => Alles ist 1024 lang
    # Falls man extra absichern will:
    labels = labels[:MAX_LENGTH]
    input_ids = input_ids[:MAX_LENGTH]
    attention_mask = attention_mask[:MAX_LENGTH]

    # 4) Rückgabe
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }
'''
def tokenize_fn(example):
    # Combine input and output (with reasoning already integrated in output) into one text
    input_text = example["input"]
    output_text = example["output"]
    combined_text = f"Input: {input_text}\nOutput: {output_text}"

    # Tokenize the combined text with specified settings
    encoded = tokenizer(
        combined_text,
        max_length=1024,
        padding="max_length",
        truncation=True
    )

    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]

    # Prepare labels: copy of input_ids
    labels = input_ids.copy()

    # Determine how many tokens belong to the prompt (the "Input: ...\nOutput: " part)
    prompt_text = f"Input: {input_text}\nOutput: "
    prompt_ids = tokenizer(
        prompt_text,
        max_length=1024,
        truncation=True,
        padding=False,
        add_special_tokens=False
    )["input_ids"]
    prompt_len = len(prompt_ids)

    # Set label IDs for the prompt part to -100 to avoid computing loss on them
    labels[:prompt_len] = [-100] * prompt_len

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }
# ==========================
# Datensatz verarbeiten
# ==========================
train_ds = train_ds.map(tokenize_fn, remove_columns=train_ds.column_names)
val_ds = val_ds.map(tokenize_fn, remove_columns=val_ds.column_names)

Map:   0%|          | 0/612 [00:00<?, ? examples/s]

Map:   0%|          | 0/154 [00:00<?, ? examples/s]

In [ ]:
# ==========================
# 6) Trainingskonfiguration
# ==========================
# - Kleine Lernrate z.B. 5e-5, um Overfitting zu reduzieren.
# - gradient_accumulation_steps=16 => simulierte Batch 16 * per_device_train_batch_size
# - Wir überwachen die Validierungsperformance (perplexity).
# - Cosine Scheduler, Warmup ratio 0.03, um Smooth Start/End zu haben.

training_args = TrainingArguments(
    output_dir="./lora-lamini-1.5b-checkpoints",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,   # => effektive Batch-Größe 32
    num_train_epochs=4,              # wir probieren 4 Epochen
    learning_rate=5e-5,              # etwas niedriger für den kleinen Datensatz
    fp16=True,
    logging_steps=50,
    evaluation_strategy="steps",
    eval_steps=200,                  # alle 200 Steps evaluieren wir
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="paged_adamw_8bit",
    report_to="none"
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
# ==========================
# 7) Trainer + Perplexity-Metrik
# ==========================
import math
def compute_metrics(eval_preds):
    # Standard: eval_preds => (logits, labels)
    logits, labels = eval_preds
    # => logits: [batch_size, seq_length, vocab_size]
    # => labels: [batch_size, seq_length]
    # berechnen wir cross entropy loss => perplexity
    predictions = torch.from_numpy(logits)
    label_ids = torch.from_numpy(labels)

    # wir maskieren positions= -100 in labels
    loss_fct = torch.nn.CrossEntropyLoss(ignore_index=-100)
    shift_logits = predictions[..., :-1, :].contiguous()
    shift_labels = label_ids[..., 1:].contiguous()
    # => z.B. next token prediction

    loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
    perplexity = math.exp(loss.item())
    return {"perplexity": perplexity}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=lambda features: {
        "input_ids": torch.tensor([f["input_ids"] for f in features]),
        "attention_mask": torch.tensor([f["attention_mask"] for f in features]),
        "labels": torch.tensor([f["labels"] for f in features]),
    }
    )

In [ ]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss


TrainOutput(global_step=76, training_loss=3.9547936288934005, metrics={'train_runtime': 3569.0942, 'train_samples_per_second': 0.686, 'train_steps_per_second': 0.021, 'total_flos': 2.15702458073088e+16, 'train_loss': 3.9547936288934005, 'epoch': 3.8366013071895426})

In [ ]:
# ==========================
# 9) Inferenz-Test
# ==========================
model.eval()
test_prompt = "PROMPT:\nSchreibe diesen Artikel um: Wir sind als nächstes dran, wenn wir uns nicht wappnen. Was bedeutet Trumps Stopp von US-Militärhilfen für die Ukraine? Militärexperte Frank Sauer erklärt, was dadurch alles wegbricht, wie lange die Ukraine noch durchhalten kann - und was das für den Rest Europas bedeutet.tagesschau.de: Die USA haben ihre Militärhilfen für die Ukraine eingestellt. Was heißt das kurzfristig für die Ukraine? Wie lange kann sie jetzt noch durchhalten?Frank Sauer: Es gab vergangenes Jahr bereits die Phase, in der die US-Hilfen aufgrund von Trumps Einwirken auf die Republikaner lange im Kongress festhingen und nichts mehr geliefert wurde. Das hatte keine sofortigen und unmittelbaren Auswirkungen auf das Gefechtsfeld in der Ukraine.Aber mittelfristig schlug sich das natürlich nieder, etwa im Bereich der Luftverteidigung. Konkret sind dann wegen der ausbleibenden Hilfe unter anderem in ukrainischen Städten durch den anhaltenden russischen Raketen- und Drohnenbeschuss mehr Zivilisten gestorben oder verwundet worden.Das an die Ukraine zu liefernde US-Material ist übrigens bezahlt und liegt jetzt ungenutzt herum. Dem US-Steuerzahler nutzt diese Aktion rein gar nichts. Sie nutzt nur Putin, der aktuell weniger denn je einen Anlass hat, sich um Frieden zu bemühen. Trump steht fest an Russlands Seite gegen die Ukraine. Und wir im restlichen Europa sind als nächstes dran, wenn wir uns nicht mit aller Kraft wappnen..\n\nREASONING:\n"

input_ids = tokenizer(test_prompt, return_tensors="pt")["input_ids"].to(device)
with torch.no_grad():
    gen_output = model.generate(
        input_ids=input_ids,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
text_out = tokenizer.decode(gen_output[0], skip_special_tokens=True)
print("================================")
print("GENERATED TEXT:")
print(text_out[len(test_prompt):])  # reasoning + article

Token indices sequence length is longer than the specified maximum sequence length for this model (544 > 512). Running this sequence through the model will result in indexing errors


GENERATED TEXT:

The article is a summary of a conversation between Frank Sauer and Trumps supporters. It does not provide any direct evidence to support the claim that Trump and his supporters have been involved in the Ukraine conflict.

The article does not mention any evidence of the Ukraine conflict, but rather discusses the alleged involvement of the USA in the conflict.

The article does not provide any evidence to support the claim that Trump and his supporters have been involved in the Ukraine conflict. However, it does mention that the US-Militär-Hilfen was responsible for the Ukraine, which is not entirely accurate.


In [ ]:
trainer.model.save_pretrained("./lamini-1.5b-lora-adapter")
tokenizer.save_pretrained("./lamini-1.5b-lora-adapter")

('./lamini-1.5b-lora-adapter/tokenizer_config.json',
 './lamini-1.5b-lora-adapter/special_tokens_map.json',
 './lamini-1.5b-lora-adapter/vocab.json',
 './lamini-1.5b-lora-adapter/merges.txt',
 './lamini-1.5b-lora-adapter/added_tokens.json',
 './lamini-1.5b-lora-adapter/tokenizer.json')